In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType
import numpy as np
from models.fcm import Dfcm
from utils.validity import * 
import time 

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("FCM_PySpark").getOrCreate()
spark

24/08/24 09:51:40 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.1.10.127 instead (on interface enp2s0)
24/08/24 09:51:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/24 09:51:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Bước 1: Đọc và chuẩn bị dữ liệu
csv_file_path = "data/csv/602_Dry_Bean.csv"
data = spark.read.csv(csv_file_path, header=True, inferSchema=True)
data = data.drop(data.columns[-1])
for column in data.columns:
    data = data.withColumn(column, col(column).cast(FloatType()))


# Bước 1: Tính tổng các giá trị thuộc tính của một đối tượng và thêm giá trị tổng này thành cột mới vào tập dữ liệu. Thực hiện trên toàn bộ dữ liệu 
data = data.withColumn("sum", sum(col(column) for column in data.columns)) 

# Bước 2: Sắp xếp tập dữ liệu theo cột tổng mới tạo theo thứ tự tăng dần 
data = data.orderBy("sum", ascending=True) 

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col, sum as _sum, row_number

# Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau, ở đây chia làm 3 dataframe 
window = Window.orderBy("sum")
data_indexed = data.withColumn("row_index", row_number().over(window))

# Define the number of clusters (segments)
k = 7  

# Tính toán số lượng dòng của tập dữ liệu để phân đoạn 
total_rows = data_indexed.count()
segment_size = total_rows // k

# Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau
segments = []
for i in range(k):
    start_idx = i * segment_size + 1
    end_idx = (i + 1) * segment_size
    if i == k - 1:  # Chắc chắn rằng tất cả các dòng đều được chia hết cho k
        end_idx = total_rows

    segment = data_indexed.filter((col("row_index") >= start_idx) & (col("row_index") <= end_idx)).drop("row_index")
    segments.append(segment)

In [5]:
# Bước 4: Đối với mỗi phân đoạn, tính tổng các cột thuộc tính (không bao gồm các cột đã tạo ở bước 1)
# tính giá trị trung bình của nó và đặt vào trong hàng mới. Hàng mới này thực sự là 
# một trong những trọng tâm cụm đã khởi tạo

# Khởi tạo một list rỗng để lưu trữ các trọng tâm cụm   
cluster_centers = []

# Duyệt qua từng phân đoạn
for segment in segments:
    # Tính tổng các cột thuộc tính 
    segment_sums = segment.agg(*[_sum(col_name).alias(col_name) for col_name in data.columns[:-1]])
    
    # Tính giá trị trung bình của các cột thuộc tính
    row_count = segment.count()
    cluster_center = [segment_sums.select(col_name).first()[0] / row_count for col_name in data.columns[:-1]]
    
    # Thêm giá trị trung bình của các cột thuộc tính vào hàng mới
    cluster_centers.append(cluster_center)

print("Initialized Cluster Centers using Naive Sharding:")
for idx, center in enumerate(cluster_centers):
    print(f"Cluster Center {idx + 1}: {center}", len(center))

24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 0

Initialized Cluster Centers using Naive Sharding:
Cluster Center 1: [28558.063271604937, 627.6085157983097, 231.46536613589942, 157.4768252313873, 1.4740293447863417, 0.726910183888404, 28910.287551440328, 190.48517128273292, 0.7543452168075145, 0.9877772662068101, 0.9100435849877051, 0.8241778463125229, 0.008139591697132053, 0.002323911700530913, 0.6802876877747936, 0.9969107743827894] 16
Cluster Center 2: [35079.62911522634, 692.5201254008729, 250.40639496634526, 179.3981036747434, 1.4059109188646937, 0.6799413533383069, 35470.3878600823, 211.29418306389954, 0.7602280726955261, 0.9889780956899188, 0.9199026766934513, 0.8467082817245413, 0.007146461028893145, 0.002280873354702111, 0.7194997052443616, 0.9973320992579185] 16
Cluster Center 3: [39753.88683127572, 740.7907338083526, 269.5134383700022, 189.1404129781841, 1.4367636770868497, 0.6935339272175551, 40195.483024691355, 224.95083723342958, 0.7574489885028989, 0.9890248735009888, 0.9117440551274107, 0.838373947566674, 0.0067816221

24/08/24 09:52:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:52:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:52:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/08/24 09:52:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [6]:
assembler = VectorAssembler(inputCols=data.columns[:-1], outputCol="features")
data = assembler.transform(data)

data_rdd = data.select("features").rdd.map(lambda row: row[0].toArray())

In [7]:
# Sau khi khởi tạo trọng tâm cụm với thuật toán sharding, sử dụng tâm cụm này là một biến toàn cục và được phát tới tất cả
# các nút công nhân (workers) thông qua broadcast trong Spark.
cluster_centers = spark.sparkContext.broadcast(cluster_centers)

In [8]:
# np.array(cluster_centers.value).shape   

In [9]:
# chia dữ liệu theo chiều ngang để phân bổ cho các nút công nhân
partitioned_data_rdd = data_rdd.repartition(k)
print("Số lượng phân vùng: ", partitioned_data_rdd.getNumPartitions())  

# # Optional: If you want to see the partition distribution, collect some samples
# samples = partitioned_data_rdd.mapPartitions(lambda it: [len(list(it))]).collect()
# print(f"Data distribution across partitions: {samples}")

Số lượng phân vùng:  7


---

## Khởi tạo tâm cụm

In [ ]:
# Lấy dữ liệu trong từng phân vùng
partitioned_data = partitioned_data_rdd.glom().collect()

fcm = Dfcm()

_start_time = time.time()

v = np.array(cluster_centers.value)
for step in range(10000):
    v_old = v.copy() 
    Us = []
    for i, partition in enumerate(partitioned_data):
        data_np = np.array(partition)
        # print(data_np.shape)

        sdistances = norm_distances(data_np, v)
        membership = fcm.update_membership_matrix(sdistances)
        Us.append(membership)
    U_final = np.concatenate(Us)

    v = fcm.update_cluster_centers(np.array(data_rdd.collect()), U_final)
    print((np.abs(v - v_old)).max(axis=(0, 1)))
    if (np.abs(v - v_old)).max(axis=(0, 1)) < 1e-5:
        break
    
metric_nt = {
    'Time': round_float(time.time() - _start_time),
    'PC': partition_coefficient(U_final) ,
}
print(metric_nt)    

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, FloatType
import numpy as np

# Step 1: Initialize Spark session
spark = SparkSession.builder \
    .appName("FCM_PySpark") \
    .getOrCreate()

# Step 2: Read data from HDFS into RDD
# For demonstration, let's create a synthetic dataset
data = [
    (0, [0.1, 0.2, 0.3]),
    (1, [0.4, 0.5, 0.6]),
    (2, [0.7, 0.8, 0.9]),
    (3, [0.2, 0.1, 0.4])
]

# Convert list to RDD
rdd = spark.sparkContext.parallelize(data)

V = np.array([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6]
])


# Broadcast initial cluster centers V to all worker nodes
cluster_centers = spark.sparkContext.broadcast(V)

# Step 4: Compute membership matrix U in parallel on each worker node
def compute_membership_matrix(pixel, centers, m=2):
    pixel = np.array(pixel)
    centers = np.array(centers)
    distances = np.linalg.norm(centers - pixel, axis=1)
    distances = np.maximum(distances, 1e-10)  # Prevent division by zero
    u = 1 / (distances ** (2 / (m - 1)))
    u /= np.sum(u)
    return u.tolist()

compute_memberships_udf = udf(lambda features: compute_membership_matrix(features, cluster_centers.value), ArrayType(FloatType()))

# Step 5: Update cluster centers V based on the new membership matrix U
def update_cluster_centers(features, memberships, k):
    features = np.array(features)
    memberships = np.array(memberships)
    new_centers = []

    for i in range(k):
        membership = memberships[:, i]
        numerator = np.sum(membership[:, np.newaxis] * features, axis=0)
        denominator = np.sum(membership)
        new_centers.append(numerator / denominator)

    return np.array(new_centers)

# Convert RDD to DataFrame to use UDFs
df = rdd.toDF(["id", "features"])

# Step 6: Iteratively update V and U until convergence or max iterations
maxLoop = 100
tolerance = 1e-5
t = 0

while t < maxLoop:
    # Compute memberships for each feature vector
    df = df.withColumn("memberships", compute_memberships_udf(col("features")))

    # Collect the features and memberships for each cluster
    features = np.array(df.select("features").rdd.map(lambda row: row[0]).collect())
    memberships = np.array(df.select("memberships").rdd.map(lambda row: row[0]).collect())

    # Update cluster centers V
    V_new = update_cluster_centers(features, memberships, k)

    # Check for convergence
    print(f"Iteration {t + 1}: {np.linalg.norm(V_new - cluster_centers.value)}")    
    if np.linalg.norm(V_new - cluster_centers.value) < tolerance:
        break

    # Update broadcast variable
    cluster_centers.unpersist()
    cluster_centers = spark.sparkContext.broadcast(V_new)
    
    t += 1

# Step 7: Output the final membership matrix U and cluster centers V
final_U = df.select("id", "memberships").collect()
final_V = cluster_centers.value

print("Final Membership Matrix U:")
for row in final_U:
    print(row)

print("\nFinal Cluster Centers V:")
print(final_V)

# Stop Spark session
spark.stop()


Iteration 1: 0.23804020760023123


Iteration 2: 0.0


Final Membership Matrix U:
Row(id=0, memberships=[1.0, 3.7037037776455956e-20])
Row(id=1, memberships=[3.7037037776455956e-20, 1.0])
Row(id=2, memberships=[0.20000000298023224, 0.800000011920929])
Row(id=3, memberships=[0.8888888955116272, 0.1111111119389534])

Final Cluster Centers V:
[[0.2        0.21489362 0.4       ]
 [0.51395349 0.60232558 0.71395349]]
